# How good are the students' topic labels?

The students' transformer labeled every ACLED event that the trigger-word rules left `unknown`
(`data/filtered_events_class_with_predicted_students.csv`). This notebook checks those labels in three steps:

1. **Score** them against 200 manually labeled random `unknown` events.
2. **Review** every error by hand. Is the prediction really wrong, or is it defensible?
3. **Explain** the real errors. Each issue is tested on all ~73,000 unknown events, not just on the examples.

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)

MANUAL_CSV   = "../data/manual_labelled_data/random_unknown_labeled.csv"
STUDENTS_CSV = "../data/filtered_events_class_with_predicted_students.csv"
REVIEW_CSV   = "students_error_review.csv"

students = pd.read_csv(STUDENTS_CSV, usecols=["event_id_cnty", "event_date", "notes", "clean_notes", "class", "predicted_class"],
                       low_memory=False)
students["date"] = pd.to_datetime(students["event_date"], format="%d %B %Y")
unknown = students[students["class"] == "unknown"]            # events the model had to label

ev = pd.read_csv(MANUAL_CSV, usecols=["event_id_cnty", "manual_label", "manual_label_alt", "notes"])
ev = ev.merge(students[["event_id_cnty", "date", "predicted_class"]], on="event_id_cnty", how="left")
assert ev["predicted_class"].notna().all()
print(f"{len(ev)} manually labeled events, {len(unknown):,} unknown events in the students' file")

200 manually labeled events, 72,984 unknown events in the students' file


## 1. Score against the manual labels

**Strict**: the prediction equals the manual label.
**Lenient**: the prediction equals the manual label or its alternative, or forms one of the accepted pairs below.
`public services` and `discrimination` are umbrellas for their specific topics. Two specific topics under the
same umbrella never count as each other.

In [2]:
ACCEPTED_PAIRS = {frozenset(p) for p in [
    ("climate", "environment"),
    ("unjust law enforcement", "blm"),
    ("public services", "health care"), ("public services", "education"), ("public services", "housing"),
    ("discrimination", "women rights"), ("discrimination", "lgbtq"),      ("discrimination", "blm"),
]}

def gold_labels(row):
    golds = {row["manual_label"]}
    if isinstance(row["manual_label_alt"], str):
        golds |= {x.strip() for x in row["manual_label_alt"].split("|")}
    return golds

ev["strict"]  = ev["predicted_class"] == ev["manual_label"]
ev["lenient"] = [any(g == p or frozenset((g, p)) in ACCEPTED_PAIRS for g in gold_labels(r))
                 for (_, r), p in zip(ev.iterrows(), ev["predicted_class"])]

print(f"strict  {ev['strict'].mean():.1%}   ({ev['strict'].sum()}/{len(ev)})")
print(f"lenient {ev['lenient'].mean():.1%}   ({ev['lenient'].sum()}/{len(ev)})")

strict  55.0%   (110/200)
lenient 68.5%   (137/200)


In [3]:
(ev.groupby("manual_label")
   .agg(events=("lenient", "size"), strict=("strict", "mean"), lenient=("lenient", "mean"))
   .sort_values("events", ascending=False)
   .style.format({"strict": "{:.0%}", "lenient": "{:.0%}"}))

,events,strict,lenient
manual_label,,,
policies & politics,65,55%,69%
labor rights,50,50%,58%
public services,17,71%,82%
environment,16,56%,69%
climate,13,38%,69%
education,8,88%,100%
immigration,5,40%,40%
unjust law enforcement,5,40%,40%
health care,4,50%,50%


## 2. How wrong are the errors really?

Every event that fails lenient scoring was read and given one verdict (`students_error_review.csv`):

- **wrong**: the prediction does not fit the note, and the manual label does.
- **defensible**: the prediction is a reasonable reading of the note. Often the manual label is only one of several fair choices.
- **no fitting class**: none of the 20 topics fits. The manual label is a catch-all, usually `policies & politics`, so the error means little.

In [4]:
review = pd.read_csv(REVIEW_CSV)
errors = ev[~ev["lenient"]].merge(review, on="event_id_cnty", how="left")
assert errors["verdict"].notna().all() and len(errors) == len(review), "review file out of sync with the errors"

print(errors["verdict"].value_counts().to_string(), "\n")

fits    = len(ev) - (errors["verdict"] == "no fitting class").sum()
correct = ev["lenient"].sum() + (errors["verdict"] == "defensible").sum()
print(f"Lenient accuracy:                                        {ev['lenient'].mean():.1%}")
print(f"Counting defensible as right, excluding no-fitting-class: {correct / fits:.1%}  ({correct}/{fits})")

verdict
wrong               35
defensible          18
no fitting class    10 

Lenient accuracy:                                        68.5%
Counting defensible as right, excluding no-fitting-class: 81.6%  (155/190)


In [5]:
with pd.option_context("display.max_rows", None):
    display(errors.sort_values(["verdict", "manual_label"])
                  [["event_id_cnty", "manual_label", "predicted_class", "verdict", "reason"]]
                  .set_index("event_id_cnty"))

,manual_label,predicted_class,verdict,reason
event_id_cnty,,,,
FRA27718,environment,policies & politics,defensible,protest is about government inaction after a factory fire
DEU8511,environment,public services,defensible,preserving a railway embankment; public services is a fair reading
SRB3046,environment,policies & politics,defensible,against local government building plans on green space
BGR624,farmers,animal welfare,defensible,pig owners against a cull; animal welfare is a fair secondary reading
GBR7537,health care,policies & politics,defensible,asks the council to reinstate funding
DEU1148,immigration,housing,defensible,about a refugee accommodation; housing is a partial reading
ESP8792,immigration,policies & politics,defensible,demand to reopen national borders is a policy demand
ROU2691,labor rights,discrimination,defensible,the note itself calls the pay exclusion discriminatory
DEU22462,labor rights,policies & politics,defensible,against a city plan to cut jobs; policy reading is fair


So roughly one in six events is labeled truly wrong. The sections below explain the 35 truly wrong
predictions. Most of them come from a few mechanisms.

## 3. Issue: the model learned the trigger words, not the topics

The training labels come from trigger-word rules, and those trigger words stay in the model's input text.
On the keyword-labeled events the students' model agrees with the rules almost perfectly. On the manual sample,
where no trigger word occurs, it gets about half right.

In [6]:
labeled = students[~students["class"].isin(["unknown", "NoN"])]
print(f"agreement with trigger-word label on keyword-labeled events: {(labeled['predicted_class'] == labeled['class']).mean():.1%}")
print(f"agreement with manual label on unknown events (strict):      {ev['strict'].mean():.1%}")

agreement with trigger-word label on keyword-labeled events: 98.5%
agreement with manual label on unknown events (strict):      55.0%


With no trigger word to find, the model latches onto ordinary words that happened to co-occur with triggers.
The table checks this on **all unknown events**. It compares how often a class is predicted when a word is in the note
versus when it is not. Some events containing these words really do belong to that class, so the gap is an upper bound.
The manual errors in the last column show the gap is not only genuine topic.

In [7]:
SHORTCUTS = [   # (word pattern, class it pulls toward, manual errors it explains)
    (r"\bstudent",                        "education",              "DEU131, NLD2455, DEU19674"),
    (r"\btraffic\b|\bbridge\b|\bairport\b", "public services",        "GBR6490, ITA3230"),
    (r"\bstation\b",                      "public services",        "FRA14109"),
    (r"\bpolice officer",                  "unjust law enforcement", "FRA3442"),
    (r"\bdiesel\b|\bfuel\b|\boil\b",       "climate",                "ITA16034"),
    (r"\bstrike\b",                       "labor rights",           "ITA18739"),
    (r"\bapartment\b",                    "housing",                "ITA10785"),
]
text = unknown["clean_notes"].fillna("")
rows = []
for pattern, cls, examples in SHORTCUTS:
    has = text.str.contains(pattern, regex=True)
    rows.append({"word": pattern.replace("\\b", ""), "pulls toward": cls, "unknown events with word": has.sum(),
                 "predicted, with word": (unknown.loc[has, "predicted_class"] == cls).mean(),
                 "predicted, without": (unknown.loc[~has, "predicted_class"] == cls).mean(),
                 "manual errors": examples})
(pd.DataFrame(rows).set_index("word")
   .style.format({"predicted, with word": "{:.0%}", "predicted, without": "{:.0%}"}))

,pulls toward,unknown events with word,"predicted, with word","predicted, without",manual errors
word,,,,,
student,education,3829,77%,2%,"DEU131, NLD2455, DEU19674"
traffic|bridge|airport,public services,3093,57%,10%,"GBR6490, ITA3230"
station,public services,1433,49%,11%,FRA14109
police officer,unjust law enforcement,2037,24%,1%,FRA3442
diesel|fuel|oil,climate,1569,30%,1%,ITA16034
strike,labor rights,2716,26%,12%,ITA18739
apartment,housing,254,66%,2%,ITA10785


Examples:

- **DEU131**: a Fridays for Future protest against a coal project is labeled `education` because students took part.
- **FRA14109**: foundry workers blocking a train station are labeled `public services`, since "railway station" is a public-services trigger.
- **ITA16034**: fishers protesting diesel prices are labeled `climate`, since "fossil fuels" is a climate trigger.
- **ITA18739**: an anarchist on hunger strike in prison is labeled `labor rights`.

## 4. Issue: a trigger-word bug makes "labour" an animal-welfare word

In the students' rules, the trigger `"labour conditions"` sits under **animal welfare**. The labor-rights triggers
only use the American spelling "labor". As a result, every training row with "labour conditions" is animal welfare,
and British "labour" almost never appears in labor-rights rows.

In [8]:
print("keyword-labeled rows containing 'labour conditions':",
      labeled.loc[labeled["notes"].str.lower().str.contains("labour conditions", na=False), "class"].value_counts().to_dict())

has = text.str.contains(r"\blabour\b", regex=True)
print(f"\nunknown events predicted animal welfare: with 'labour' {(unknown.loc[has, 'predicted_class'] == 'animal welfare').mean():.1%} "
      f"(n={has.sum()}), without {(unknown.loc[~has, 'predicted_class'] == 'animal welfare').mean():.1%}")

errors.loc[errors["event_id_cnty"].isin(["ESP7468", "GBR8215"]), ["event_id_cnty", "predicted_class", "notes"]]

keyword-labeled rows containing 'labour conditions': {'animal welfare': 46}

unknown events predicted animal welfare: with 'labour' 24.3% (n=786), without 1.6%


,event_id_cnty,predicted_class,notes
32,ESP7468,animal welfare,"On 8 November 2021, hairdressers protested in Aranda del Duero (Burgos, Castilla y Leon), calling on the government ..."
58,GBR8215,animal welfare,"On 5 February 2025, in the afternoon, at the call of JNIV, veterans of The Troubles marched through the city center ..."


Hairdressers asking for lower VAT on their "labour sector", and veterans protesting "Labour's plans", both become
animal-welfare protests. About a quarter of unknown events mentioning "labour" are labeled this way. The rule was later
fixed in `pipeline/finetuner/triggers.py`, but the students' labels predate that fix.

## 5. Issue: `policies & politics` is a catch-all, and some events fit no class at all

The taxonomy has no class for peace and military, history and remembrance, crime and safety, or foreign affairs.
Such events get `policies & politics` from the manual labeler, and often from the model too.

In [9]:
pred_share = unknown["predicted_class"].value_counts(normalize=True)
print(f"share of unknown events the students label policies & politics: {pred_share['policies & politics']:.1%}")
print("by year:", unknown.groupby(unknown["date"].dt.year)["predicted_class"]
      .apply(lambda s: f"{(s == 'policies & politics').mean():.0%}").to_dict())

gov = text.str.contains(r"\bmunicipal|\bcouncil\b|\bcity hall\b|\bmayor\b", regex=True)
print(f"\npredicted policies & politics: with a local-government word {(unknown.loc[gov, 'predicted_class'] == 'policies & politics').mean():.0%}, "
      f"without {(unknown.loc[~gov, 'predicted_class'] == 'policies & politics').mean():.0%}")

pp = ev["predicted_class"] == "policies & politics"
print(f"on the manual sample: predicted {pp.sum()} times, lenient-correct {ev.loc[pp, 'lenient'].sum()} times")

share of unknown events the students label policies & politics: 35.5%
by year: {2018: '58%', 2019: '59%', 2020: '39%', 2021: '35%', 2022: '33%', 2023: '30%', 2024: '34%', 2025: '31%'}

predicted policies & politics: with a local-government word 59%, without 34%
on the manual sample: predicted 56 times, lenient-correct 37 times


A third of all unknown events end up in `policies & politics`, and more than half of the unknown events from 2018–2019 do.
The class is right about two times in three when predicted. Many labor disputes with a council or government also land
there: GBR2295, MKD981, ESP10467. The 10 **no fitting class** events above show the flip side. A hostage journalist,
nuclear weapons or a mafia commemoration cannot be labeled correctly with these 20 topics.

## 6. Issue: the training sample covers only recent years for most big classes

The students' balancing notebook keeps the **first N rows per class** (`groupby('class').head(N)`). Their data file is
sorted **newest first**, so each large class keeps only its most recent events. N was 3000 or 6000; both are shown.

In [10]:
rows = []
for cls, g in labeled.groupby("class"):
    rows.append({"class": cls, "keyword rows": len(g),
                 "all rows from": g["date"].min().year,
                 "N=3000 from": g.head(3000)["date"].min().year,
                 "N=6000 from": g.head(6000)["date"].min().year})
spans = pd.DataFrame(rows).set_index("class").sort_values("keyword rows", ascending=False)
spans

,keyword rows,all rows from,N=3000 from,N=6000 from
class,,,,
pandemic,20684,2020,2022,2021
labor rights,19357,2018,2024,2023
farmers,10047,2018,2024,2023
palestine-israel conflict,7632,2018,2024,2023
education,7445,2018,2023,2021
women rights,6514,2018,2022,2020
ukraine-russia war,6074,2018,2022,2020
climate,5551,2019,2022,2019
environment,4742,2018,2022,2018


With N=3000, `labor rights` training data is only 2024–2025, `farmers` only 2024–2025, and `pandemic` mostly 2022,
after the lockdown protests. The model never saw what an older labor dispute or a 2020 Covid protest looks like
in those classes.

This is a real flaw, but the manual sample cannot show how much it costs:

In [11]:
lr = ev[ev["manual_label"] == "labor rights"]
print(lr.groupby(lr["date"].dt.year >= 2024)["lenient"].agg(events="size", lenient="mean")
        .rename(index={False: "before 2024", True: "2024 or later"}).to_string())

               events   lenient
date                           
before 2024        42  0.571429
2024 or later       8  0.625000


Accuracy on labor events is only slightly higher for 2024 and later, and there are just 8 such events. The date
bias is likely part of why labor rights is weak, but this sample cannot prove it. The effect should be tested with a
larger or date-stratified manual sample.

## Summary

| | |
|---|---|
| Strict accuracy | 55.0% |
| Lenient accuracy | 68.5% |
| Counting defensible as right, excluding events with no fitting class | ~82% |

Of the 63 lenient errors, 35 are truly wrong, 18 are defensible, and 10 fit no class. The truly wrong ones
come mainly from:

1. **Trigger-word shortcuts.** The model learned the rules' vocabulary, so "student", "station", "traffic",
   "diesel" or "strike" decide the topic regardless of what the protest is about.
2. **The "labour conditions" trigger bug.** British "labour" pulls events toward animal welfare.
3. **A catch-all class and taxonomy gaps.** A third of unknown events become `policies & politics`, including
   labor disputes with public employers.
4. **Newest-first training sample.** Most big classes only contain their most recent events. This is a
   plausible cause of the weak labor rights scores, but it is not measurable on this sample.

The seven `pandemic` errors (e.g. FRA4674, EST354, DEU23622) have no clear cause in the data and remain unexplained.